In [40]:
import os
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.layers import InputLayer,Conv2D,MaxPooling2D,Flatten,Dense
from tensorflow.keras import Sequential
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from keras.layers import Layer
from keras.layers import Reshape
from keras.layers import Lambda
from keras import initializers


In [41]:
def preprocessing(path):
    img = Image.open(path)
    img_resized = img.resize((224,224))
    return np.array(img_resized)/255

In [42]:
data_path = '../data/spectrograms'
class0 = 'class_0'
class1 = 'class_1'

In [43]:
non_seizure = []
seizure = []
n_seizure_names = [f for f in os.listdir(os.path.join(data_path,class0))]
seizure_names = [f for f in os.listdir(os.path.join(data_path,class1))]
for path in n_seizure_names:
    non_seizure.append(preprocessing(os.path.join(data_path,class0,path)))
for path in seizure_names:
    seizure.append(preprocessing(os.path.join(data_path,class1,path)))

In [44]:
non_seizure_labels = [0]*len(non_seizure)
seizure_labels = [1]*len(seizure)

data = non_seizure+seizure
data_labels = non_seizure_labels+seizure_labels

X, y = shuffle(np.array(data), np.eye(2)[data_labels],random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.3,random_state = 42)
X_test, X_val, y_test, y_val = train_test_split(X_test,y_test,test_size=1/3,random_state = 42)

In [45]:
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

checkpoint = ModelCheckpoint(
    '../weights/CapsuleNET.weights.h5',
    monitor='val_loss',
    save_best_only=True,
    save_weights_only=True
)

In [46]:
import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.layers import *
from tensorflow.keras import initializers
from tensorflow.keras.models import Model

def squash(inputs):
    squared_norm = K.sum(K.square(inputs), axis=-1, keepdims=True)
    return (squared_norm / (1 + squared_norm)) / (K.sqrt(squared_norm + K.epsilon())) * inputs

class DigitCapsuleLayer(Layer):
    def __init__(self, **kwargs):
        super(DigitCapsuleLayer, self).__init__(**kwargs)
        self.kernel_initializer = initializers.get('glorot_uniform')
    
    def build(self, input_shape):
        self.W = self.add_weight(
            shape=[2, input_shape[1], 16, 8],  # 2 capsules for binary
            initializer=self.kernel_initializer,
            name='weights'
        )
        self.built = True
    
    def call(self, inputs):
        inputs_expanded = K.expand_dims(inputs, 1)
        inputs_tiled = K.tile(inputs_expanded, [1, 2, 1, 1])
        u_hat = K.map_fn(lambda x: K.batch_dot(x, self.W, [2, 3]), elems=inputs_tiled)
        b = tf.zeros(shape=[K.shape(inputs)[0], 2, K.shape(inputs)[1]])  # Fixed shape reference
        for i in range(3 - 1):
            c = tf.nn.softmax(b, axis=1)
            s = K.batch_dot(c, u_hat, [2, 2])
            v = squash(s)
            b = b + K.batch_dot(v, u_hat, [2, 3])
        return v
    
    def compute_output_shape(self, input_shape):
        return (None, 2, 16)

def output_layer(inputs):
    return K.sqrt(K.sum(K.square(inputs), -1) + K.epsilon())

def mask(outputs):
    if type(outputs) != list:
        norm_outputs = K.sqrt(K.sum(K.square(outputs), -1) + K.epsilon())
        y = K.one_hot(indices=K.argmax(norm_outputs, 1), num_classes=2)
        y = Reshape((2, 1))(y)
        return Flatten()(y * outputs)
    else:
        y = Reshape((2, 1))(outputs[1])
        masked_output = y * outputs[0]
        return Flatten()(masked_output)

def loss_fn(y_true, y_pred):
    L = y_true * K.square(K.maximum(0., 0.9 - y_pred)) + \
        0.5 * (1 - y_true) * K.square(K.maximum(0., y_pred - 0.1))
    return K.mean(K.sum(L, 1))

In [48]:
input_tensor = Input(shape=(224, 224, 4))  # This creates the input tensor

# conv layers
conv1 = Conv2D(256, (9, 9), activation='relu', padding='valid')(input_tensor)
conv2 = Conv2D(256, (9, 9), strides=2, padding='valid')(conv1)


num_capsules = 104 * 104 * 32  # 32 capsules per spatial location
reshaped = Reshape((num_capsules, 8))(conv2)
squashed_output = Lambda(squash, output_shape=lambda s: s)(reshaped)

digit_caps = DigitCapsuleLayer()(squashed_output)
outputs = Lambda(output_layer)(digit_caps)

# Fixed: Use Input() for labels as well
inputs_label = Input(shape=(2,))
masked = Lambda(mask)([digit_caps, inputs_label])
masked_for_test = Lambda(mask)(digit_caps)

# decoder
decoded_inputs = Input(shape=(16 * 2,))
dense1 = Dense(512, activation='relu')(decoded_inputs)
dense2 = Dense(1024, activation='relu')(dense1)
decoded_outputs = Dense(224 * 224 * 4, activation='sigmoid')(dense2)
decoded_outputs = Reshape((224, 224, 4))(decoded_outputs)

decoder = Model(decoded_inputs, decoded_outputs)

# Final models
model = Model([input_tensor, inputs_label], [outputs, decoder(masked)])
test_model = Model(input_tensor, [outputs, decoder(masked_for_test)])

print("Model created successfully!")
print(f"Model input shape: {model.input_shape}")
print(f"Model output shape: {model.output_shape}")

2025-05-28 15:09:13.966230: W external/local_xla/xla/tsl/framework/bfc_allocator.cc:501] Allocator (GPU_0_bfc) ran out of memory trying to allocate 338.00MiB (rounded to 354418688)requested by op AddV2
If the cause is memory fragmentation maybe the environment variable 'TF_GPU_ALLOCATOR=cuda_malloc_async' will improve the situation. 
Current allocation summary follows.
Current allocation summary follows.
2025-05-28 15:09:13.966266: I external/local_xla/xla/tsl/framework/bfc_allocator.cc:1058] BFCAllocator dump for GPU_0_bfc
2025-05-28 15:09:13.966273: I external/local_xla/xla/tsl/framework/bfc_allocator.cc:1065] Bin (256): 	Total Chunks: 20, Chunks in use: 20. 5.0KiB allocated for chunks. 5.0KiB in use in bin. 124B client-requested in use in bin.
2025-05-28 15:09:13.966275: I external/local_xla/xla/tsl/framework/bfc_allocator.cc:1065] Bin (512): 	Total Chunks: 0, Chunks in use: 0. 0B allocated for chunks. 0B in use in bin. 0B client-requested in use in bin.
2025-05-28 15:09:13.966278: 

ResourceExhaustedError: {{function_node __wrapped__AddV2_device_/job:localhost/replica:0/task:0/device:GPU:0}} failed to allocate memory [Op:AddV2] name: 

In [ ]:
model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.000001),
              loss=[loss_fn, 'mse'],
              loss_weights=[1., 0.0005],
              metrics=['accuracy'])

In [ ]:
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)
checkpoint = ModelCheckpoint(
    'weights/RNN_LSTM.weights.h5',
    monitor='val_loss',
    save_best_only=True,
    save_weights_only=True
)

In [ ]:
history = model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=30,
        batch_size = 20,
        callbacks= [early_stopping,checkpoint]
    )